# Assignment: Data Wrangling

### Reading material: `tidy_data.pdf`

**Q1.** This question provides some practice cleaning variables which have common problems.
1. For `./data/airbnb_hw.csv`, clean the `Price` variable as well as you can, and explain the choices you make. How many missing values do you end up with? (Hint: What happens to the formatting when a price goes over 999 dollars, say from 675 to 1,112?)
2. For the Minnesota police use of for data, `./data/mn_police_use_of_force.csv`, clean the `subject_injury` variable, handling the NA's; this gives a value `Yes` when a person was injured by police, and `No` when no injury occurred. What proportion of the values are missing? Is this a concern? Cross-tabulate your cleaned `subject_injury` variable with the `force_type` variable. Are there any patterns regarding when the data are missing? 

**Q2.** Go to https://sharkattackfile.net/ and download their dataset on shark attacks (Hint: `GSAF5.xls`).

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Roshan Mahesh

**Date:** 09/06/2026

In [ ]:
# Your Phase 1 solution to the problem goes here.
# Q1.1 - Airbnb Price
import pandas as pd
 
df = pd.read_csv("./data/airbnb_hw.csv", low_memory=False)
 
price = df["Price"]
print("dtype as read:", price.dtype)
print("missing as read:", price.isna().sum())
 
clean = (
    price.astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)  # harmless here, defensive
)
df["Price"] = pd.to_numeric(clean, errors="coerce")
 
print("missing after cleaning:", df["Price"].isna().sum())
print(df["Price"].describe())
print("listings above $999:", (df["Price"] > 999).sum())
 
 # Q1.2 - MN police subject_injury
import pandas as pd
 
mn = pd.read_csv("./data/mn_police_use_of_force.csv", low_memory=False)
 
print("raw value counts:")
print(mn["subject_injury"].value_counts(dropna=False))
 
# Normalize case/whitespace, then map only clear Yes/No values.
norm = mn["subject_injury"].astype(str).str.strip().str.lower()
mn["subject_injury"] = norm.map({"yes": "Yes", "no": "No"})
 
n_missing = mn["subject_injury"].isna().sum()
print("missing:", n_missing)
print("proportion missing:", round(n_missing / len(mn), 4))
 
# Cross-tabulation with force_type, keeping missing as its own category.
xtab = pd.crosstab(
    mn["force_type"],
    mn["subject_injury"].fillna("Missing"),
    margins=True,
)
print("\ncounts:\n", xtab)
 
# Row proportions make the missingness pattern easier to read.
rates = pd.crosstab(
    mn["force_type"],
    mn["subject_injury"].fillna("Missing"),
    normalize="index",
).round(3)
print("\nrow proportions:\n", rates)


###### Question 2 ######

import re
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
 
# 2.1
# pandas needs the xlrd engine for it (openpyxl only reads .xlsx).
df = pd.read_excel("./data/GSAF5.xls", engine="xlrd")
print("shape:", df.shape)
 
 
# 2.2
print("\nnon-null counts:\n", df.notna().sum())
 
empty = [c for c in df.columns if df[c].notna().sum() == 0]
junk = [c for c in df.columns if df[c].notna().sum() < 0.01 * len(df)]
print("\nfully empty:", empty)
print("effectively empty:", junk)
 
df = df.drop(columns=junk)
print("shape after drop:", df.shape)
 
 
# 2.3
year = pd.to_numeric(df["Year"], errors="coerce")
print("\nyear range:", year.min(), "to", year.max())
print("zeros (unknown date):", (year == 0).sum())
print("before 1500:", ((year > 0) & (year < 1500)).sum())
 
df["Year"] = year.replace(0, pd.NA) 
 
recent = df[df["Year"] >= 1940].copy()
print("attacks since 1940:", len(recent))
 
per_year = recent["Year"].value_counts().sort_index()
print("\nattacks per decade since 1940:")
print(recent.groupby((recent["Year"] // 10 * 10).astype(int)).size())
 
plt.figure(figsize=(9, 4))
per_year.plot()
plt.title("Recorded shark attacks per year, 1940 onward")
plt.xlabel("Year")
plt.ylabel("Attacks")
plt.tight_layout()
plt.savefig("attacks_by_year.png", dpi=120)
plt.close()
 
 
# 2.4
raw_age = df["Age"].astype(str).str.strip()
 
age = pd.to_numeric(raw_age, errors="coerce")
 
# Recover decade forms: "20s", "30's", "40+" -> 20, 30, 40.
decade = raw_age.str.match(r"^\d{2}(s|'s|\+)$", na=False)
age = age.where(~decade, pd.to_numeric(raw_age.str.extract(r"^(\d{2})")[0]))
 
# Anything still non-numeric is genuinely ambiguous ("Teen", "?",
# "45 and 15", "28 & 22") and stays missing.
age = age.where((age > 0) & (age < 110))
 
df["Age"] = age
print("\nage missing:", age.isna().sum(), "of", len(df))
print(age.describe())
 
plt.figure(figsize=(8, 4))
plt.hist(age.dropna(), bins=range(0, 101, 5), edgecolor="white")
plt.title("Ages of shark attack victims")
plt.xlabel("Age")
plt.ylabel("Number of victims")
plt.tight_layout()
plt.savefig("age_histogram.png", dpi=120)
plt.close()
 
 
# 2.5
sex = df["Sex"].astype(str).str.strip().str.upper()
df["Sex"] = sex.map({"M": "M", "F": "F"}) 
counts = df["Sex"].value_counts()
print("\nsex counts:\n", counts)
print("proportion male:", round(counts["M"] / counts.sum(), 4))
 
 
# 2.6
t = df["Type"].astype(str).str.strip().str.lower()
df["Type"] = t.map({"unprovoked": "Unprovoked", "provoked": "Provoked"}).fillna(
    "Unknown"
)
tc = df["Type"].value_counts()
print("\ntype counts:\n", tc)
print("proportion unprovoked:", round(tc["Unprovoked"] / len(df), 4))
 
 


dtype as read: str
missing as read: 0
missing if converted naively: 181
missing after cleaning: 0
count    30478.000000
mean       163.589737
std        197.785454
min         10.000000
25%         80.000000
50%        125.000000
75%        195.000000
max      10000.000000
Name: Price, dtype: float64
listings above $999: 181
raw value counts:
subject_injury
NaN    9848
Yes    1631
No     1446
Name: count, dtype: int64
missing: 9848
proportion missing: 0.7619

counts:
 subject_injury               Missing    No   Yes    All
force_type                                             
Baton                              2     0     2      4
Bodily Force                    7051  1093  1286   9430
Chemical Irritant               1421   131    41   1593
Firearm                            0     2     0      2
Gun Point Display                 27    33    44    104
Improvised Weapon                 74    34    40    148
Less Lethal                       87     0     0     87
Less Lethal Projectile 

Matplotlib is building the font cache; this may take a moment.


shape: (7117, 23)

non-null counts:
 Date              7117
Year              7115
Type              7099
Country           7067
State             6630
Location          6550
Activity          6534
Name              6899
Sex               6539
Age               4123
Injury            7081
Fatal Y/N         6556
Time              3590
Species           3986
Source            7097
pdf               6799
href formula      6794
href              6796
Case Number       6798
Case Number.1     6797
original order    6799
Unnamed: 21          1
Unnamed: 22          2
dtype: int64

fully empty: []
effectively empty: ['Unnamed: 21', 'Unnamed: 22']
shape after drop: (7117, 21)

year range: 0.0 to 2026.0
zeros (unknown date): 129
before 1500: 3
attacks since 1940: 5581

attacks per decade since 1940:
Year
1940     283
1950     467
1960     618
1970     339
1980     438
1990     572
2000    1022
2010    1249
2020     593
dtype: int64

age missing: 3124 of 7117
count    3993.000000
mean       28.280

In [ ]:
#Question 1
# 1. Price is a string not a number, so that is why the formatting breaks at $1000. The prices for four digit numbers consists of a comma, which is not a valid character for a number. The solution is to clean the data by removing the commas and dollar signs, and then converting the cleaned strings to numeric values.
# 1. Missing values after cleaning: 0 out of 30,478.
# 2. The cleaning decision worth explaining: blanks are mapped to NaN, not to No. 76% of subject_injury values are missing, which is serious because it ranges from 0% for Less Lethal Projectile to 40.3% for K9 bites, 75.4% for Tasers, and 100% for Maximal Restraint Technique, so the recorded injury rates are computed on a pattern.

#Question 2
# 3. 5,581 attacks, which rise from 283 in the 1940s to 1,249 in the 2010s, so recorded attacks are increasing
# 5. 5,713 victims are male and 818 female, giving a proportion male of 0.8748
# 6. 5,263 Unprovoked, 649 Provoked, and 1,205 Unknown, so the proportion unprovoked is 0.7395


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 691a97087fd6d9247fae204bfdf6bdfe24da7f20
- **AI tool and model used:** ChatGPT 5.6 Sol

### *Prompts used

1. Refine my existing code for better following this assignments intructions. First provide a list of changes, then provide the code base.
2. Are there any final changes I should add to be sure?

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]
1. Removed repeated imports and notebook-unfriendly matplotlib.use("Agg").
2. Used pandas’ nullable string type so missing values remain missing during cleaning.
3. Cleaned Airbnb prices by removing commas and dollar signs before numeric conversion.
3. Added direct answers: 0 missing prices and 181 listings above $999.
4. Made the police missingness result explicit: 76.19% is missing, which is concerning because missingness varies substantially by force type.
5. Replaced the arbitrary “less than 1% populated” column rule with targeted removal of empty and Unnamed columns.
6. Fixed age cleaning so approximate or multi-person entries are not treated as exact ages.
7. Displayed graphs inside the notebook with plt.show().
8. Reindexed annual attack counts so years with zero observations would still appear.
9. Added direct interpretations for the year trend, male proportion, and unprovoked proportion.
10. Clarified denominators: male percentage uses known male/female records, while unprovoked percentage includes Unknown as required.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification                                                                                                                            |
| ------ | -------- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1      | Accept   | Imports only need to appear once, and removing `matplotlib.use("Agg")` allows plots to display normally in the notebook.                           |
| 2      | Accept   | Pandas’ nullable string type preserves missing values instead of converting them into the text `"nan"`.                                            |
| 3      | Accept   | Removing commas and dollar signs allows prices such as `"1,112"` to be converted correctly to numbers.                                             |
| 4      | Accept   | The cleaned output confirms that there are 0 missing prices and 181 listings priced above $999.                                                    |
| 5      | Accept   | The output shows that 76.19% of `subject_injury` values are missing and that missingness varies by force type.                                     |
| 6      | Accept   | Removing empty and `Unnamed` columns is more targeted. The `Unnamed` columns contain spreadsheet notes rather than useful variables.               |
| 7      | Accept   | Entries such as `"Teen"`, `"30s"`, and `"28 & 22"` do not provide one exact age, so leaving them missing avoids inventing values.                  |
| 8      | Accept   | `plt.show()` displays the required graphs directly inside the notebook.                                                                            |
| 9      | Accept   | Reindexing ensures that years with zero recorded attacks still appear in the graph.                                                                |
| 10     | Accept   | The assignment asks for conclusions, so written interpretations directly answer each question.                                                     |
| 11     | Accept   | The male percentage uses known male/female records, while the unprovoked percentage includes all attacks because `Unknown` is a required category. |


### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [4]:
# Your Phase 2 solution to the problem goes here.

import matplotlib.pyplot as plt
import pandas as pd


############################
# Q1.1 - Airbnb Price
############################

airbnb = pd.read_csv("./data/airbnb_hw.csv", low_memory=False)

print("Original Price dtype:", airbnb["Price"].dtype)
print("Original missing values:", airbnb["Price"].isna().sum())

# Prices above $999 contain commas, such as "1,112".
# Remove currency formatting before converting the column to numeric.
price_clean = (
    airbnb["Price"]
    .astype("string")
    .str.strip()
    .str.replace(r"[,$]", "", regex=True)
)

airbnb["Price"] = pd.to_numeric(price_clean, errors="coerce")

price_missing = airbnb["Price"].isna().sum()
prices_above_999 = (airbnb["Price"] > 999).sum()

print("\nCleaned Price summary:")
print(airbnb["Price"].describe())

print("\nMissing prices after cleaning:", price_missing)
print("Listings with prices above $999:", prices_above_999)

print(
    "\nAnswer: Removing commas and currency symbols preserves prices above "
    "$999 instead of turning them into missing values. The cleaned Price "
    f"column has {price_missing} missing values. There are "
    f"{prices_above_999} listings priced above $999."
)


############################
# Q1.2 - MN police subject_injury
############################

mn = pd.read_csv(
    "./data/mn_police_use_of_force.csv",
    low_memory=False
)

print("\nRaw subject_injury values:")
print(mn["subject_injury"].value_counts(dropna=False))

# Normalize capitalization and whitespace.
# Values other than clear Yes/No responses remain missing.
injury_normalized = (
    mn["subject_injury"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

mn["subject_injury"] = injury_normalized.map(
    {"yes": "Yes", "no": "No"}
)

injury_missing = mn["subject_injury"].isna().sum()
injury_missing_proportion = mn["subject_injury"].isna().mean()

print("\nMissing subject_injury values:", injury_missing)
print(
    "Proportion missing:",
    f"{injury_missing_proportion:.2%}"
)

# Treat missing injury data as its own category for the cross-tabulation.
injury_for_table = mn["subject_injury"].fillna("Missing")
force_for_table = mn["force_type"].fillna("Missing force type")

injury_counts = pd.crosstab(
    force_for_table,
    injury_for_table,
    margins=True
)

injury_row_proportions = pd.crosstab(
    force_for_table,
    injury_for_table,
    normalize="index"
).mul(100).round(1)

print("\nCounts by force type:")
print(injury_counts)

print("\nRow percentages by force type:")
print(injury_row_proportions)

print(
    "\nAnswer: 76.19% of subject_injury values are missing. This is a "
    "concern because most observations do not indicate whether an injury "
    "occurred. Missingness also varies by force type: it is 100% for Less "
    "Lethal and Maximal Restraint Technique, 89.2% for Chemical Irritant, "
    "and 26.0% for Gun Point Display. Therefore, the missing data do not "
    "appear evenly distributed across force types."
)


############################
# Q2.1 - Open the shark attack file
############################

sharks = pd.read_excel(
    "./data/GSAF5.xls",
    engine="xlrd"
)

# Remove accidental whitespace from column names, such as "Species ".
sharks.columns = sharks.columns.str.strip()

print("\nOriginal shark attack data shape:", sharks.shape)


############################
# Q2.2 - Drop columns without meaningful data
############################

print("\nNonmissing values per column:")
print(sharks.notna().sum())

# Drop completely empty columns and spreadsheet artifact columns.
# The Unnamed columns contain isolated editing notes rather than variables.
empty_columns = sharks.columns[sharks.isna().all()].tolist()
unnamed_columns = [
    column
    for column in sharks.columns
    if column.startswith("Unnamed")
]

columns_to_drop = list(
    dict.fromkeys(empty_columns + unnamed_columns)
)

print("\nColumns being dropped:", columns_to_drop)

sharks = sharks.drop(columns=columns_to_drop)

print("Shape after dropping columns:", sharks.shape)


############################
# Q2.3 - Clean Year and examine the trend
############################

year_numeric = pd.to_numeric(
    sharks["Year"],
    errors="coerce"
)

print("\nRaw year range:")
print("Minimum:", year_numeric.min())
print("Maximum:", year_numeric.max())
print("Values recorded as zero:", (year_numeric == 0).sum())

# A year of zero represents an unknown year rather than a real date.
sharks["Year"] = year_numeric.mask(year_numeric == 0)

print("\nKnown year range after treating zero as missing:")
print("Minimum:", sharks["Year"].min())
print("Maximum:", sharks["Year"].max())
print("Missing years:", sharks["Year"].isna().sum())

# Restrict the analysis to attacks from 1940 onward.
recent_attacks = sharks.loc[sharks["Year"] >= 1940].copy()
recent_attacks["Year"] = recent_attacks["Year"].astype(int)

print("Attacks from 1940 onward:", len(recent_attacks))

# Include every year in the range, even if a year had no recorded attacks.
year_range = range(
    1940,
    recent_attacks["Year"].max() + 1
)

attacks_per_year = (
    recent_attacks.groupby("Year")
    .size()
    .reindex(year_range, fill_value=0)
)

attacks_per_decade = recent_attacks.groupby(
    recent_attacks["Year"] // 10 * 10
).size()

print("\nAttacks per decade:")
print(attacks_per_decade)

plt.figure(figsize=(10, 5))

plt.plot(
    attacks_per_year.index,
    attacks_per_year.values,
    color="steelblue",
    alpha=0.55,
    label="Annual count"
)

plt.plot(
    attacks_per_year.index,
    attacks_per_year.rolling(5, center=True).mean(),
    color="darkred",
    linewidth=2,
    label="Five-year moving average"
)

plt.title("Recorded Shark Attacks per Year, 1940 Onward")
plt.xlabel("Year")
plt.ylabel("Number of recorded attacks")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(
    "\nAnswer: Recorded attacks generally increase over time, especially "
    "from the 1990s through the 2010s. The 2020s should not be compared "
    "directly with complete decades because the current decade is incomplete."
)


############################
# Q2.4 - Clean Age and create a histogram
############################

raw_age = (
    sharks["Age"]
    .astype("string")
    .str.strip()
)

# Convert exact numeric ages.
# Ambiguous values such as "Teen", "30s", "8 or 10", and entries containing
# multiple victims remain missing because they do not provide one exact age.
age_numeric = pd.to_numeric(
    raw_age,
    errors="coerce"
)

# Keep only plausible human ages.
sharks["Age"] = age_numeric.where(
    age_numeric.between(0, 110)
)

print("\nCleaned Age summary:")
print(sharks["Age"].describe())
print(
    "Missing or ambiguous ages:",
    sharks["Age"].isna().sum(),
    "of",
    len(sharks)
)

plt.figure(figsize=(9, 5))

plt.hist(
    sharks["Age"].dropna(),
    bins=range(0, 91, 5),
    color="steelblue",
    edgecolor="white"
)

plt.title("Ages of Shark Attack Victims")
plt.xlabel("Age")
plt.ylabel("Number of victims")
plt.xticks(range(0, 91, 10))
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


############################
# Q2.5 - Clean Sex and find the male proportion
############################

sex_normalized = (
    sharks["Sex"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sharks["Sex"] = sex_normalized.map(
    {"M": "Male", "F": "Female"}
)

sex_counts = sharks["Sex"].value_counts()
male_proportion = (
    sex_counts["Male"] / sex_counts.sum()
)

print("\nCleaned sex counts:")
print(sex_counts)

print(
    "Proportion male among records with a known male/female value:",
    f"{male_proportion:.2%}"
)

print(
    f"\nAnswer: {male_proportion:.2%} of victims with a known "
    "male/female value are male."
)


############################
# Q2.6 - Clean Type and find the unprovoked proportion
############################

type_normalized = (
    sharks["Type"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# All categories other than Provoked and Unprovoked become Unknown.
sharks["Type"] = (
    type_normalized
    .map(
        {
            "provoked": "Provoked",
            "unprovoked": "Unprovoked",
        }
    )
    .fillna("Unknown")
)

type_counts = sharks["Type"].value_counts()
unprovoked_proportion = (
    sharks["Type"] == "Unprovoked"
).mean()

print("\nCleaned attack type counts:")
print(type_counts)

print(
    "Proportion of all attacks that were unprovoked:",
    f"{unprovoked_proportion:.2%}"
)

print(
    f"\nAnswer: {unprovoked_proportion:.2%} of all recorded attacks "
    "are classified as unprovoked."
)


Original Price dtype: str
Original missing values: 0

Cleaned Price summary:
count       30478.0
mean     163.589737
std      197.785454
min            10.0
25%            80.0
50%           125.0
75%           195.0
max         10000.0
Name: Price, dtype: Float64

Missing prices after cleaning: 0
Listings with prices above $999: 181

Answer: Removing commas and currency symbols preserves prices above $999 instead of turning them into missing values. The cleaned Price column has 0 missing values. There are 181 listings priced above $999.

Raw subject_injury values:
subject_injury
NaN    9848
Yes    1631
No     1446
Name: count, dtype: int64

Missing subject_injury values: 9848
Proportion missing: 76.19%

Counts by force type:
subject_injury               Missing    No   Yes    All
force_type                                             
Baton                              2     0     2      4
Bodily Force                    7051  1093  1286   9430
Chemical Irritant               1421   1

/var/folders/nj/cgs31jvd1kn3xrr2gv2fc3pr0000gn/T/ipykernel_26469/2200927410.py:223: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/nj/cgs31jvd1kn3xrr2gv2fc3pr0000gn/T/ipykernel_26469/2200927410.py:279: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The AI generated code improves my solution by making the cleaning choices more efficient and answering each question more directly. For the Airbnb data, it removes commas and dollar signs before converting prices to numbers, which correctly preserves prices over $999. For the police data, it keeps missing values intact, calculates the missing proportion, and uses cross-tabulations to show that missingness varies by force type. For the shark attack data, it removes only empty columns instead of relying on an 1% cutoff, displays the graphs in the notebook, and includes years with zero recorded attacks. One issue I noticed with the original AI suggestion was that it converted approximate ages such as 30s into exact ages, which creates false precision. I changed this so unclear ages remain missing. I verified the final solution by running the code on all three datasets and checking the summaries and cross-tabulations. The final numbers came out to 0 missing Airbnb prices, 76.19% missing police injury values, 87.48% male victims among known records, and 73.95% unprovoked attacks. My main remaining concerns are the large amounts of missing police injury data, the messy age entries, potential reporting bias in early shark records, and the incomplete numbers for the 2020s.